problem: Identify active accounts where usage in the last 30 days dropped by 50% or more compared to the previous 30 days.

In [2]:
%pip install pandas

  Using cached pandas-3.1.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (16 kB)
  Using cached numpy-2.5.5-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached pandas-3.1.1-cp314-cp314-macosx_11_0_arm64.whl (11.3 MB)
Using cached numpy-2.5.5-cp314-cp314-macosx_14_0_arm64.whl (5.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [6]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("saas_usage.db")

In [7]:
pd.read_sql("""
WITH usage_summary AS (
    SELECT
        account_id,
        SUM(CASE
            WHEN event_date >= DATE('now', '-30 days')
            AND event_date < DATE('now')
            THEN usage_amount
            ELSE 0
        END) AS last_30_days_usage,
        SUM(CASE
            WHEN event_date >= DATE('now', '-60 days')
            AND event_date < DATE('now', '-30 days')
            THEN usage_amount
            ELSE 0
        END) AS previous_30_days_usage
    FROM usage_events
    WHERE account_status = 'active'
    GROUP BY account_id
)
SELECT
    account_id,
    last_30_days_usage,
    previous_30_days_usage,
    ROUND(
        100.0 * (previous_30_days_usage - last_30_days_usage)
        / previous_30_days_usage, 2
    ) AS percentage_drop
FROM usage_summary
WHERE previous_30_days_usage > 0
    AND last_30_days_usage <= previous_30_days_usage * 0.5
ORDER BY percentage_drop DESC;
""", conn)

DatabaseError: Execution failed on sql '
WITH usage_summary AS (
    SELECT
        account_id,
        SUM(CASE
            WHEN event_date >= DATE('now', '-30 days')
            AND event_date < DATE('now')
            THEN usage_amount
            ELSE 0
        END) AS last_30_days_usage,
        SUM(CASE
            WHEN event_date >= DATE('now', '-60 days')
            AND event_date < DATE('now', '-30 days')
            THEN usage_amount
            ELSE 0
        END) AS previous_30_days_usage
    FROM usage_events
    WHERE account_status = 'active'
    GROUP BY account_id
)
SELECT
    account_id,
    last_30_days_usage,
    previous_30_days_usage,
    ROUND(
        100.0 * (previous_30_days_usage - last_30_days_usage)
        / previous_30_days_usage, 2
    ) AS percentage_drop
FROM usage_summary
WHERE previous_30_days_usage > 0
    AND last_30_days_usage <= previous_30_days_usage * 0.5
ORDER BY percentage_drop DESC;
': no such column: usage_amount

In [9]:
pd.read_sql("""
WITH usage_summary AS (
    SELECT
        u.account_id,
        SUM(CASE
            WHEN u.event_date >= r.recent_30_day_start
            AND u.event_date <= r.reporting_date
            THEN 1 ELSE 0
        END) AS last_30_days_usage,
        SUM(CASE
            WHEN u.event_date >= r.recent_60_day_start
            AND u.event_date < r.recent_30_day_start
            THEN 1 ELSE 0
        END) AS previous_30_days_usage
    FROM usage_events AS u
    JOIN accounts AS a
        ON u.account_id = a.account_id
    CROSS JOIN reporting_context AS r
    WHERE a.account_status = 'active'
    GROUP BY u.account_id
)
SELECT
    account_id,
    last_30_days_usage,
    previous_30_days_usage,
    ROUND(
        100.0 * (previous_30_days_usage - last_30_days_usage)
        / previous_30_days_usage, 2
    ) AS percentage_drop
FROM usage_summary
WHERE previous_30_days_usage > 0
    AND last_30_days_usage <= previous_30_days_usage * 0.5
ORDER BY percentage_drop DESC;
""", conn)

,account_id,last_30_days_usage,previous_30_days_usage,percentage_drop
0,3,25,75,66.67
1,7,20,60,66.67
2,1,30,80,62.50
3,9,20,50,60.00
4,11,20,40,50.00


In [10]:
pd.read_sql("""
WITH usage_summary AS (
    SELECT
        u.account_id,
        SUM(CASE
            WHEN u.event_date >= r.recent_30_day_start
            AND u.event_date <= r.reporting_date
            THEN 1 ELSE 0
        END) AS last_30_days,
        SUM(CASE
            WHEN u.event_date >= r.recent_60_day_start
            AND u.event_date < r.recent_30_day_start
            THEN 1 ELSE 0
        END) AS previous_30_days
    FROM usage_events AS u
    CROSS JOIN reporting_context AS r
    GROUP BY u.account_id
)
SELECT
    a.account_id,
    a.account_name,
    u.last_30_days,
    u.previous_30_days,
    u.previous_30_days - u.last_30_days AS usage_drop,
    ROUND(
        100.0 * (u.previous_30_days - u.last_30_days)
        / u.previous_30_days, 2
    ) AS percentage_drop
FROM usage_summary AS u
JOIN accounts AS a
    ON u.account_id = a.account_id
WHERE a.account_status = 'active'
    AND u.previous_30_days > 0
    AND u.last_30_days <= u.previous_30_days * 0.5
ORDER BY percentage_drop DESC, usage_drop DESC;
""", conn)

,account_id,account_name,last_30_days,previous_30_days,usage_drop,percentage_drop
0,3,Account_3,25,75,50,66.67
1,7,Account_7,20,60,40,66.67
2,1,Account_1,30,80,50,62.50
3,9,Account_9,20,50,30,60.00
4,11,Account_11,20,40,20,50.00
